In [ ]:
# ── CELL 1: Setup & Dataset Download (CPU OK) ─────────────────────────────────
#
# How your Drive should look BEFORE running this cell:
#   MyDrive/inpainting/main.py
#   MyDrive/inpainting/configs/config.py
#   MyDrive/inpainting/colab_setup.py
#   MyDrive/inpainting/requirements.txt
#   ... (the full repo, flat upload)
#
# Flow:
#   1. Mount Drive, bootstrap colab_setup.py from Drive → init()
#      (copies full repo to /content/inpainting/, restores checkpoints/results)
#   2. Install requirements.txt  (or fall back to inline package list)
#   3. Download datasets (idempotent — skips if already downloaded)
#   4. GPU sanity check
#
# Run on CPU runtime first to download data, then switch to A100 for Cell 2.

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

# ── Step 1: Mount Drive + bootstrap colab_setup before repo is local ──────────
try:
    from google.colab import drive as _gd  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        _gd.mount("/content/drive")

    _drive_cs = Path("/content/drive/MyDrive/inpainting/colab_setup.py")
    if not _drive_cs.exists():
        raise FileNotFoundError(
            f"colab_setup.py not found at {_drive_cs}\n"
            "Make sure you uploaded the full repo to MyDrive/inpainting/"
        )

    # Load colab_setup from Drive before the repo is local
    shutil.copy(str(_drive_cs), "/tmp/_cs_boot.py")
    _spec = importlib.util.spec_from_file_location("colab_setup", "/tmp/_cs_boot.py")
    _cs   = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_cs)

    LOCAL_ROOT = _cs.LOCAL_ROOT
    LOCAL_DATA = _cs.LOCAL_DATA
    DRIVE_ROOT = _cs.DRIVE_ROOT
    _cs.init()   # copies repo, restores checkpoints + results from Drive tarballs

except ImportError:
    # Not in Colab — assume running locally (dev / testing)
    print("Not in Colab. Using local directory as LOCAL_ROOT.")
    LOCAL_ROOT = Path(".")
    LOCAL_DATA = Path("datasets")
    DRIVE_ROOT = None
    _cs        = None

# Now the full repo is at LOCAL_ROOT
if str(LOCAL_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_ROOT))
os.chdir(str(LOCAL_ROOT))

# ── Step 2: Install requirements ──────────────────────────────────────────────
_REQ_FALLBACK = [
    # Core ML
    "torch>=2.0.0", "torchvision>=0.15.0", "timm>=0.9.0",
    # Vision / metrics
    "lpips>=0.1.4", "clean-fid>=0.1.35",
    # Data / utils
    "Pillow>=9.5.0", "scikit-image>=0.21.0", "scipy>=1.10.0",
    "numpy>=1.24.0", "einops>=0.6.0", "omegaconf>=2.3.0",
    # HuggingFace (CelebA-HQ download)
    "huggingface_hub>=0.26.0", "datasets>=2.18.0",
    # Other
    "gdown>=5.0.0", "matplotlib>=3.7.0", "seaborn>=0.12.0", "pandas>=2.0.0",
]

req = LOCAL_ROOT / "requirements.txt"
if req.exists():
    print("\nInstalling requirements from requirements.txt ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    print("Requirements installed.")
else:
    print(
        "\nWARNING: requirements.txt not found in repo copy.\n"
        "  Make sure requirements.txt is uploaded to MyDrive/inpainting/\n"
        "  Falling back to inline package list..."
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q"] + _REQ_FALLBACK
    )
    print("Fallback packages installed.")

# ── Step 3: Download datasets ──────────────────────────────────────────────────
print("\nChecking / downloading datasets...")
from data.download import download_all

drive_cache = (
    str(DRIVE_ROOT / "datasets")
    if DRIVE_ROOT is not None and (DRIVE_ROOT / "datasets").exists()
    else None
)
download_all(base_dir=str(LOCAL_DATA), drive_cache=drive_cache)

# ── Step 4: GPU check ──────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU: {name} ({vram:.1f} GB VRAM)")
    if "A100" in name:
        print("  A100 detected — optimal hardware.")
    elif vram >= 16:
        print("  >= 16 GB VRAM — OK, but A100 is faster.")
    else:
        print("  WARNING: low VRAM. Switch runtime to A100 before Cell 2.")
else:
    print("\nNo GPU detected. Switch to A100 GPU runtime before running Cell 2.")
    print("  Runtime > Change runtime type > A100 GPU")


In [ ]:
# ── CELL 2: Main Training Run (A100 required) ──────────────────────────────────
#
# Optimised run matrix:
#   - Phases 1-3 SKIPPED (resnet34 + attn_skip + gated_conv locked in)
#   - Phase 4: 6 configs × 35 epochs  (~8.75h on A100-40GB)
#   - Multi-seed L0+L4: 4 extra runs × 35 epochs (~8.75h)
#   - ConvNeXt L0+L4: 2 runs × 35 epochs (~4.4h)
#   - Eval + cross-domain: ~30 min
#   Total: ~22h — fits in one Colab session if you start at the top of the hour.
#
# Safe to interrupt + re-run — resumes via pipeline_state.json.
# Auto-saves to Drive every 20 min (single-tarball, atomic writes).

import json
import os
import subprocess
import sys
from pathlib import Path

from colab_setup import LOCAL_ROOT, LOCAL_DATA

os.chdir(str(LOCAL_ROOT))
if str(LOCAL_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_ROOT))

from colab_setup import auto_save, init, require_data, save, status

init()                      # idempotent: restores any new Drive tarballs
auto_save(interval_min=20)  # background daemon
require_data()              # fail-fast: Places365 must be present
status()

# ── Launch run.py ─────────────────────────────────────────────────────────────
# Flags:
#   --skip_download   → datasets already downloaded in Cell 1
#   --skip_phases123  → skip backbone/skip/conv selection (already validated)
#   Phases 1-3 skipped by default in run.py; pass --run_phases123 to re-enable.
proc = subprocess.Popen(
    [
        sys.executable, "-u", "run.py",
        "--profile",     "production",
        "--skip_download",
        # Remove the line below if you want to re-run phases 1-3:
        # "--run_phases123",
        # Add --skip_multiseed or --skip_convnext if you need to save more time.
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(LOCAL_ROOT),
)

try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print("\nInterrupted. Re-run Cell 2 to resume from checkpoint.")

return_code = proc.wait()
print(f"\nrun.py exited with code {return_code}")

# ── Final save ────────────────────────────────────────────────────────────────
save("final", force_all=True)
status()

# ── Summary table ─────────────────────────────────────────────────────────────
results_path = LOCAL_ROOT / "results" / "all_results.json"
if results_path.exists():
    with open(results_path) as f:
        r = json.load(f)
    p4 = r.get("phase4_eval", {})
    print("\n" + "=" * 72)
    print(f"{'Config':<28}  {'PSNR':>6}  {'SSIM':>6}  {'LPIPS':>7}  {'B-MAE':>7}  {'FID':>7}")
    print("-" * 72)
    for name in ["L0_base","L1_boundary_uniform","L2_boundary_grad",
                 "L3_spectral_only","L3c_adaptive_spectral","L4_full_method"]:
        if name in p4:
            g = p4[name].get("global", {})
            print(
                f"  {name:<26}  "
                f"{g.get('psnr',0):6.2f}  {g.get('ssim',0):6.4f}  "
                f"{g.get('lpips',-1):7.4f}  {g.get('boundary_mae',0):7.4f}  "
                f"{g.get('fid',-1):7.1f}"
            )
    print("=" * 72)


In [ ]:
# ── CELL 3: Smoke Test (optional — ~5 min on A100) ────────────────────────────
#
# Quick sanity check before the full 24h run.
# Verifies:
#   1. Loss decreases over 5 epochs
#   2. ASBC band weights diverge from uniform (learning is happening)
#   3. Model output is valid (correct shape, no NaN)
#
# Run this BEFORE Cell 2 to catch bugs early.

import os
import sys
from pathlib import Path

from colab_setup import LOCAL_ROOT

os.chdir(str(LOCAL_ROOT))
if str(LOCAL_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_ROOT))

import torch
from configs.config import get_smoke_config
from data.dataset import setup_data
from losses.losses import InpaintingLossManager
from main import make_model_cfg, seed_everything
from models.architecture import BoundaryAwareInpainter
from training.trainer import enable_a100_flags, run_experiment

enable_a100_flags()
seed_everything(42)

cfg    = get_smoke_config()
cfg.make_dirs()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Smoke test on {device}")

data      = setup_data(cfg)
model_cfg = make_model_cfg("resnet34", use_attn_skip=True, use_gated_conv=True, multiscale=True)
model     = BoundaryAwareInpainter(model_cfg)

print("\nRunning 5-epoch smoke test with full_adaptive loss...")
result = run_experiment(
    name="smoke_test",
    model=model,
    train_loader=data["fast_train_loader"],
    val_loader=data["fast_val_loader"],
    cfg=cfg,
    loss_config="full_adaptive",
    num_epochs=5,
    freeze_encoder_epochs=1,
    patience=10,
    device=device,
    val_every_n_epochs=1,
)

# ── Assertions ─────────────────────────────────────────────────────────────────
train_losses = result["history"]["train_loss"]
assert len(train_losses) >= 2, "Expected at least 2 epoch records"
first, last  = train_losses[0], train_losses[-1]
print(f"\nTrain loss: epoch 1 = {first:.4f}, epoch {len(train_losses)} = {last:.4f}")
if last < first:
    print("  PASS: loss decreased.")
else:
    print("  WARNING: loss did not decrease — check LR / data.")

# ASBC weight check
loss_manager = InpaintingLossManager(cfg.loss, device)
weights      = loss_manager.adaptive_spectral.get_band_weights()
weight_std   = float(weights.std())
print(f"\nASBC band weights: {[f'{w:.4f}' for w in weights.tolist()]}")
print(f"  std = {weight_std:.4f}  (> 0 means learning is happening)")
if weight_std > 1e-3:
    print("  PASS: ASBC weights are non-uniform.")
else:
    print("  INFO: ASBC weights still uniform — may need more epochs.")

# Output shape / NaN check
model.eval()
with torch.no_grad():
    dummy_img   = torch.rand(1, 3, 256, 256).to(device)
    dummy_mask  = (torch.rand(1, 1, 256, 256) > 0.5).float().to(device)
    dummy_bound = torch.rand(1, 1, 256, 256).to(device)
    out = model(dummy_img * (1 - dummy_mask), dummy_mask, dummy_bound)["output"]

assert out.shape == (1, 3, 256, 256), f"Unexpected output shape: {out.shape}"
assert not torch.isnan(out).any(), "NaN in model output!"
print(f"\nOutput shape: {out.shape}  PASS")
print("\nSmoke test complete. Proceed to Cell 2 for the full pipeline.")


In [ ]:
# ── CELL 4: Figures & Tables (CPU OK — run after Cell 2 completes) ────────────
#
# Loads results/all_results.json and generates paper-ready figures.
# All outputs saved to LOCAL_ROOT/results/.
#
# Tables printed:
#   - Ablation (L0–L4 + L3c) on Places365
#   - Cross-domain (CelebA-HQ + DTD)
#   - Multi-seed stats (L0/L4, 3 seeds)

import json
import os
import sys
from pathlib import Path

from colab_setup import LOCAL_ROOT

os.chdir(str(LOCAL_ROOT))
if str(LOCAL_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_ROOT))

import matplotlib
matplotlib.use("Agg")

results_file = LOCAL_ROOT / "results" / "all_results.json"
if not results_file.exists():
    print("results/all_results.json not found. Run Cell 2 first.")
else:
    with open(results_file) as f:
        r = json.load(f)

    out_dir = LOCAL_ROOT / "results"
    p4      = r.get("phase4_eval",  {})
    cel     = r.get("celeba_eval",  {})
    dtd     = r.get("dtd_eval",     {})
    seeds   = r.get("seed_stats",   {})

    # ── Ablation table ────────────────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("ABLATION TABLE  (Places365)")
    print(f"{'Config':<28}  {'PSNR':>6}  {'SSIM':>6}  {'LPIPS':>7}  {'B-MAE':>7}  {'Spec':>7}  {'FID':>7}")
    print("-" * 80)
    for name in [
        "L0_base", "L1_boundary_uniform", "L2_boundary_grad",
        "L3_spectral_only", "L3c_adaptive_spectral", "L4_full_method",
    ]:
        if name in p4:
            g = p4[name].get("global", {})
            print(
                f"  {name:<26}  "
                f"{g.get('psnr', 0):6.2f}  "
                f"{g.get('ssim', 0):6.4f}  "
                f"{g.get('lpips', -1):7.4f}  "
                f"{g.get('boundary_mae', 0):7.4f}  "
                f"{g.get('spectral_coherence', 0):7.4f}  "
                f"{g.get('fid', -1):7.1f}"
            )
    print("=" * 80)

    # ── Cross-domain table ────────────────────────────────────────────────────
    if cel or dtd:
        print("\nCROSS-DOMAIN  (zero-shot, no fine-tuning)")
        print(f"{'Config':<28}  {'CelebA PSNR':>11}  {'CelebA B-MAE':>12}  {'DTD PSNR':>9}  {'DTD B-MAE':>10}")
        print("-" * 75)
        for name in ["L0_base", "L3c_adaptive_spectral", "L4_full_method"]:
            cg = cel.get(name, {})
            dg = dtd.get(name, {})
            print(
                f"  {name:<26}  "
                f"{cg.get('psnr', float('nan')):11.2f}  "
                f"{cg.get('boundary_mae', float('nan')):12.4f}  "
                f"{dg.get('psnr', float('nan')):9.2f}  "
                f"{dg.get('boundary_mae', float('nan')):10.4f}"
            )

    # ── Multi-seed table ──────────────────────────────────────────────────────
    if seeds:
        print("\nMULTI-SEED STATISTICS  (3 seeds: 42 / 1 / 2, mean ± std)")
        for exp_name in ["L0_base", "L4_full_method"]:
            if exp_name in seeds:
                print(f"  {exp_name}:")
                for k in ("psnr", "ssim", "boundary_mae", "spectral_coherence"):
                    s = seeds[exp_name].get(k)
                    if s:
                        print(f"    {k:22s}: {s['mean']:.4f} ± {s['std']:.4f}")

    # ── Training curves + ablation table figure ───────────────────────────────
    from utils.visualize import plot_ablation_table, plot_sensitivity, plot_training_curves

    curves = {}
    for name in p4.keys():
        cf = out_dir / f"phase4_{name}_curves.json"
        if cf.exists():
            with open(cf) as f:
                cd = json.load(f)
            curves[name] = {
                "train_loss": [e.get("total", 0) for e in cd.get("train", [])],
                "val_loss":   [e.get("total", 0) for e in cd.get("val",   [])],
                "lr":         cd.get("lr", []),
            }

    if curves:
        plot_training_curves(curves, out_dir, "Phase 4: Ablation Training Curves")
        print(f"\nTraining curves → {out_dir}/training_curves.png")

    try:
        plot_ablation_table(p4, out_dir)
        print(f"Ablation table  → {out_dir}/ablation_table.png")
    except Exception as e:
        print(f"Ablation table warning: {e}")

    print(f"\nAll figures saved to {out_dir}/")
